# 2PL_DQN — Google Colab 実行ノートブック

リポジトリ: <https://github.com/ituki0426/2PL_DQN>

このノートブックは `run_grid.py` / `benchmark_cost.py` / `analyze_selection.py` / `make_tables.py` を Colab から順に実行するためのものです。

**推奨設定**: `ランタイム → ランタイムのタイプを変更 → GPU`（`run_grid.py` の学習を高速化）。

**セッション制限に注意**: 無料枠は最大 12h、90 分程度アイドルで切断。replication は独立プロセスなので、切れたら該当 rep だけ再実行すれば続きから進められます。

## 1. セットアップ — リポジトリ取得と依存関係のインストール

In [ ]:
import os
if not os.path.isdir('2PL_DQN'):
    !git clone https://github.com/ituki0426/2PL_DQN.git
%cd 2PL_DQN
!pip install -q -r requirements.txt

In [ ]:
import torch, numpy, pandas, matplotlib
print('torch', torch.__version__, 'CUDA', torch.cuda.is_available())
print('numpy', numpy.__version__)
print('pandas', pandas.__version__)
print('matplotlib', matplotlib.__version__)

## 2. （任意）Google Drive をマウント

セッションが切れても結果を残したい場合に使います。使わない場合はこのセルをスキップ。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/2PL_DQN_results'
!mkdir -p "$DRIVE_OUT"
print('output dir:', DRIVE_OUT)

## 3. スモークテスト（`--quick`）

`--quick` は `n_episodes=200` に短縮して数分で完走します。まずここで環境確認。

In [ ]:
# 実験を選択（以降の全セルで使用）
EXPERIMENT = "EXP001"  # "EXP001" | "EXP002"
!python run_grid.py --list-experiments

!python run_grid.py --experiment $EXPERIMENT --grid main --rep 4 --quick --out /tmp/2PL_DQN_smoke

## 4. `run_grid.py` — 本番グリッド実行

各グリッドで走らせる replication 番号:

| グリッド | reps | 概要 |
| --- | --- | --- |
| `main` | 4, 5, 6 | 既存 vs 提案（表 tab:main） |
| `guess` | 4, 5, 6 | 3PLM 応答（表 tab:guess） |
| `sensitivity` | 1, 2, 3 | 報酬 × γ の網羅（表 tab:sensitivity） |
| `state` | 1, 2, 3 | 状態 A/B/C（表 tab:state） |
| `ablation` | 1, 2, 3 | 一因子入れ替え（表 tab:ablation） |

1 条件あたりの学習は重いです。無料 GPU で完走が難しいときは `--conditions proposed` などで条件を絞ってください。

In [ ]:
# ここで対象グリッドと rep を選ぶ
GRID = 'main'          # 'main' | 'sensitivity' | 'state' | 'ablation' | 'guess'
REPS = [4, 5, 6]        # main / guess は 4-6、それ以外は 1-3
EXTRA_ARGS = ''         # 例: '--conditions proposed' や '--quick'

In [ ]:
for rep in REPS:
    print(f'\n=== {GRID}: rep {rep} ===', flush=True)
    !python run_grid.py --experiment $EXPERIMENT --grid $GRID --rep $rep $EXTRA_ARGS

In [ ]:
# 全 rep が揃ったら mean.csv を作る
!python run_grid.py --experiment $EXPERIMENT --grid $GRID --aggregate

## 5. `benchmark_cost.py` — 選択規則の推論コスト計測

`result/<実験名>/cost/cost.csv` を出力します。単一スレッド固定なので数分。

In [ ]:
!python benchmark_cost.py --experiment $EXPERIMENT

## 6. `analyze_selection.py` — 選択アイテムの分析＋図出力

`result/<実験名>/selection/` に CSV と `fig_selection.pdf` を出力します。

In [ ]:
!python analyze_selection.py --experiment $EXPERIMENT

In [ ]:
# PDF をノートブック内で確認したい場合
from IPython.display import IFrame
IFrame(f'result/{EXPERIMENT}/selection/fig_selection.pdf', width=800, height=500)

## 7. `make_tables.py` — LaTeX 表行の生成

対応するグリッドの `mean.csv` / `rep*.csv` が揃っている必要があります。EXP001 の計算済み結果は `result/EXP001/` に含まれています。EXP002 は実行後に利用できます。

In [ ]:
for name in ['main', 'sensitivity', 'state', 'ablation', 'guess', 'cost']:
    print(f'\n===== {name} =====')
    !python make_tables.py --experiment $EXPERIMENT $name

## 8. 結果を Google Drive に保存（マウント済みの場合）

In [ ]:
!cp -r result "$DRIVE_OUT/"
!ls "$DRIVE_OUT/result"

In [ ]:
# 選択した実験の保存先を確認
!ls result/$EXPERIMENT